# Deliverable 2 – Campus Coffee Shop Sales Pipeline

Linda Qviller Halvorsen

Marita Nyjordet Pedersen

In [25]:
## Imports
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

**Helper Function – Finding Data Files**

We created the find_data_file() helper function to locate input files without relying on hardcoded absolute file paths. This makes the project more flexible and allows it to work even when the project directory is located in a different place.

By keeping file path handling in a separate function, we avoid repeating code, improve readability, and make the project easier to maintain and share between group members.

In [26]:

def find_data_file(filename: str) -> Path:
    """Find a data file in shared_data/d2_data near the project."""
    current = Path.cwd().resolve()

    for parent in (current, *current.parents):
        candidate = (
            parent / "shared_data" / "d2_data" / filename
        )

        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Could not find {filename} in shared_data/d2_data "
        "under the current directory or its parents."
    )


**Task 1: File ingestion and regular expressions**

In [27]:
def read_text_file(file_path: Path) -> tuple[str, str]:
    """
    Read a text file using UTF-8 or ISO-8859-1 encoding.

    Args:
        file_path: Path to the text file.

    Returns:
        A tuple containing the file content and the encoding used.
    """
    # Try UTF-8 first and fall back to Latin-1 if decoding fails
    try:
        file_text_content = file_path.read_text(encoding="utf-8")
        encoding_used = "UTF-8"

    except UnicodeDecodeError:
        file_text_content = file_path.read_text(encoding="ISO-8859-1")
        encoding_used = "ISO-8859-1"

    return file_text_content, encoding_used

In [28]:
log_pattern = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>[^\]\s]+)\] "
    r"item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$",
    re.MULTILINE
)

In [29]:
def parse_log_file(file_text_content: str) -> list[dict]:
    """
    Parse valid transaction lines from log text.

    Args:
        file_text_content: The complete log file content.

    Returns:
        A list of dictionaries representing valid transactions.
    """
    transactions = []
    
    # The regex matches valid transactions and ignores system noise
    for match in log_pattern.finditer(file_text_content):
        transaction = match.groupdict()

        # Convert the captured price from string to float
        transaction["price"] = float(transaction["price"])
        transactions.append(transaction)

    return transactions

**Task 2: Recursive utility functions**

In [30]:

def find_log_files(directory: str | Path) -> list[Path]:
    """
    Recursively find all .log files in a directory and its subdirectories.

    Args:
        directory: Path to the directory to search.

    Returns:
        A list of Path objects for all matching .log files.
    """
    # Convert the input path to a Path object
    directory = Path(directory)

    # Create a list to store matching log files
    log_files = []

    # Iterate through the directory contents
    for item in directory.iterdir():

        # If the item is a directory, search recursively
        if item.is_dir():
            log_files.extend(find_log_files(item))

        # If the item is a .log file, add its Path object
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    # Return the list of Path objects
    return log_files


In [31]:

def find_active_register_ids(data: object) -> set[str]:
    """Find all active register IDs in nested dictionaries and lists."""
    register_ids: set[str] = set()

    if isinstance(data, dict):
        # Collect IDs only when the register is marked as active
        if data.get("active") is True and isinstance(data.get("id"), str):
            register_ids.add(data["id"])

        # Recursively inspect all dictionary values
        for value in data.values():
            register_ids.update(find_active_register_ids(value))

    elif isinstance(data, list):
        # Recursively inspect each list element
        for item in data:
            register_ids.update(find_active_register_ids(item))

    return register_ids


**Limitation:** Register IDs are not necessarily unique across locations. The function identifies active register IDs without distinguishing between locations. This may cause registers from different locations to be treated as the same register, limiting location-specific analysis.

**Data Loading**

We created the load_all_transactions() helper function to separate data loading from the main workflow. The function uses read_text_file() and parse_log_file() to process each log file and combine all valid transactions into a single list.

This approach avoids code duplication, improves readability, and makes the main workflow easier to maintain. It also allows us to test the data-loading process separately before creating the Pandas DataFrame.

In [32]:

def load_all_transactions(log_files: list[Path]) -> list[dict]:
    """
    Read and parse transactions from multiple log files.

    Args:
        log_files: A list of paths to log files.

    Returns:
        A combined list of transaction dictionaries.
    """
    all_transactions = []

    for log_file in log_files:
        # Read each file using its supported character encoding
        file_text_content, encoding_used = read_text_file(log_file)

        # Extract valid transaction records
        transactions = parse_log_file(file_text_content)

        # Combine records from all files into one list
        all_transactions.extend(transactions)

    return all_transactions

**Task 3: Functions for cleaning and analysing data using NumPy**

In [33]:

def remove_price_outliers(
    prices: list[float] | np.ndarray
) -> np.ndarray:
    """
    Replace price outliers using the 3-standard-deviation rule.

    Args:
        prices: A list or NumPy array of transaction prices.

    Returns:
        A cleaned 1D NumPy float array.
    """
    # Convert input to a NumPy float array
    prices_array = np.array(prices, dtype=float)

    # Check that the array is one-dimensional
    if prices_array.ndim != 1:
        raise ValueError("Prices must be a 1D array.")

    # Handle empty input
    if prices_array.size == 0:
        return prices_array

    # Calculate mean and standard deviation
    mean_price = np.mean(prices_array)
    std_price = np.std(prices_array)

    # Identify outliers
    outliers = np.abs(prices_array - mean_price) > 3 * std_price

    # Find valid prices
    valid_prices = prices_array[~outliers]

    # Replace outliers with the median of valid prices
    if valid_prices.size > 0:
        median_price = np.median(valid_prices)
        prices_array[outliers] = median_price

    return prices_array
  

**Limitation:** The 3-standard-deviation rule is applied to all transaction prices collectively, regardless of item or currency. Differences in normal price levels may therefore affect outlier detection. 

In [34]:

def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Compute the element-wise refund ratio.

    Args:
        sales_count: A NumPy array containing sales counts.
        refund_count: A NumPy array containing refund counts.

    Returns:
        A NumPy array containing the refund ratios.
    """
    # Add a small epsilon to prevent division by zero
    refund_ratio = refund_count / (sales_count + 10**-6)

    return refund_ratio

**Task 4: Data analysis using Pandas**

In [35]:

def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Convert transaction dictionaries into a DataFrame.

    Convert timestamps to datetime, set the timestamp as
    the index, and sort transactions chronologically.

    Args:
        transactions: A list of transaction dictionaries.

    Returns:
        A DataFrame sorted by timestamp.
    """
    df = pd.DataFrame(transactions)

    # Convert timestamps to datetime for time-based analysis
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Use timestamps as the DataFrame index
    df = df.set_index("timestamp")

    # Sort transactions chronologically
    df = df.sort_index()

    return df



In [36]:

def correct_nok_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Multiply NOK transaction prices by 10.

    Args:
        df: A DataFrame containing transaction prices and currencies.

    Returns:
        A DataFrame with corrected NOK prices.
    """
    # Create a copy to avoid modifying the original DataFrame
    corrected_df = df.copy()

    # Identify transactions in NOK
    nok_mask = corrected_df["currency"] == "NOK"

    # Correct NOK prices
    corrected_df.loc[nok_mask, "price"] *= 10.0

    return corrected_df


**Limitation:** The correction must only be applied once to each dataset. Repeated application would multiply NOK prices by 10 again and produce incorrect values.

In [37]:

def convert_prices_to_nok(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert transaction prices to NOK using approximate exchange rates.

    Args:
        df: A DataFrame with corrected transaction prices.

    Returns:
        A DataFrame with prices converted to NOK.
    """
    # Preserve the original DataFrame and its currencies
    converted_df = df.copy()

    # Use assumed exchange rates to make totals comparable
    exchange_rates = {
        "NOK": 1.0,
        "EUR": 10.0,
        "USD": 10.0
    }

    # Convert all prices to NOK before aggregation
    converted_df["price"] = (
        converted_df["price"]
        * converted_df["currency"].map(exchange_rates)
    )

    # Label the converted values with their common currency
    converted_df["currency"] = "NOK"

    return converted_df


**Currency conversion:** EUR and USD are converted using an assumed exchange rate of 10 NOK per currency unit. This ensures that transaction values are expressed in the same currency before aggregation, but the resulting totals are estimates rather than values based on official exchange rates. The original currencies are preserved in the cleaned transaction dataset.

In [38]:

def calculate_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate hourly total price per item.

    Args:
        df: A transaction DataFrame indexed by timestamp.

    Returns:
        A DataFrame containing hourly revenue per item.
    """
    # Aggregate all transaction statuses in one-hour intervals
    hourly_revenue = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .reset_index()
    )

    return hourly_revenue


**Limitation:** This function sums prices across all transaction statuses (SUCCESS, REFUND, and CANCELLED). Therefore, the results represent aggregated transaction values rather than actual net revenue.

In [39]:
def create_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table of total transaction prices
    per item across status categories.

    Args:
        df: A transaction DataFrame containing
            item, price, and status columns.

    Returns:
        A pivot table with items as rows and
        status categories as columns.
    """
    # Sum transaction values separately for each status and item
    pivot = pd.pivot_table(
        df,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

    # Keep a consistent status order, including missing categories
    pivot = pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )

    return pivot


In [40]:
def calculate_refund_ratio_by_item(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Calculate the refund ratio for each item.

    Args:
        df: A DataFrame containing item and status columns.

    Returns:
        A DataFrame containing sales count, refund count,
        and refund ratio for each item.
    """
    items = sorted(df["item"].unique())

    # Count successful transactions rather than monetary values
    sales_count = (
        df[df["status"] == "SUCCESS"]
        .groupby("item")
        .size()
        .reindex(items, fill_value=0)
    )

    # Count refunded transactions for the same items
    refund_count = (
        df[df["status"] == "REFUND"]
        .groupby("item")
        .size()
        .reindex(items, fill_value=0)
    )

    refund_report = pd.DataFrame({
        "sales_count": sales_count,
        "refund_count": refund_count
    })

    # Reuse the NumPy function from Task 3
    refund_report["refund_ratio"] = compute_refund_ratio(
        refund_report["sales_count"].to_numpy(),
        refund_report["refund_count"].to_numpy()
    )

    refund_report.index.name = "item"

    return refund_report

**Task 5: Create full workflow**

In [41]:

def run_workflow() -> None:
    """
    Run the complete transaction processing workflow.

    Load transactions, clean prices, correct NOK errors,
    convert currencies for reporting, and export results.
    """

    # Locate data directory and log files
    data_directory = find_data_file("store_config.json").parent
    log_files = find_log_files(data_directory)

    # Load store configuration
    with open(
        data_directory / "store_config.json",
        "r",
        encoding="utf-8"
    ) as config_file:
        store_config = json.load(config_file)

    active_register_ids = find_active_register_ids(store_config)

    print("Log files found:", len(log_files))
    print("Active register IDs:", sorted(active_register_ids))

    # Load and parse transactions
    all_transactions = load_all_transactions(log_files)

    print("Transactions parsed:", len(all_transactions))

    if not all_transactions:
        raise ValueError("No valid transactions were found.")

    # Create transaction DataFrame
    df = create_transaction_dataframe(all_transactions)

    # Remove price outliers
    df["price"] = remove_price_outliers(
        df["price"].to_numpy(copy=True)
    )

    # Correct incorrectly registered NOK prices
    df = correct_nok_prices(df)

    # Convert prices to NOK for reporting
    df_nok = convert_prices_to_nok(df)

    # Generate reports using NOK values
    hourly_revenue = calculate_hourly_revenue(df_nok)
    revenue_pivot = create_revenue_pivot(df_nok)
    refund_report = calculate_refund_ratio_by_item(df_nok)

    # Create output directory
    output_directory = Path.cwd() / "output"
    output_directory.mkdir(parents=True, exist_ok=True)

    # Export cleaned transactions with original currencies
    df.to_csv(
        output_directory / "cleaned_transactions.csv"
    )

    # Export reports calculated in NOK
    hourly_revenue.to_json(
        output_directory / "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=4
    )

    revenue_pivot.to_csv(
        output_directory / "status_pivot_table.csv"
    )

    refund_report.to_csv(
        output_directory / "item_refund_ratios.csv"
    )

    print("Workflow completed successfully.")
    print("Output directory:", output_directory)



**Run Full Workflow**

In [42]:
run_workflow()

Log files found: 9
Active register IDs: ['POS_01', 'POS_02', 'POS_03']
Transactions parsed: 720
Workflow completed successfully.
Output directory: c:\Users\Bruker\INF_prosjekter\Deliverable_02_week_43\output


**Testing and Validation**

The following tests verify the individual functions and their integration.
The tests use separate variables to avoid modifying the original transaction data.
We use assert statements to verify that the functions produce expected results.
If all assertions pass, a PASS message is displayed. If a test fails, Python raises an AssertionError.
Passing tests confirms correct behavior for the tested cases, but does not guarantee error-free code.

In [43]:

# Test file reading and log parsing
test_log_path = find_data_file(
    "2026/library_cafe/POS_01/register_sales.log"
)

test_content, test_encoding = read_text_file(test_log_path)
test_transactions = parse_log_file(test_content)

assert isinstance(test_content, str)
assert test_encoding in ("UTF-8", "ISO-8859-1")
assert isinstance(test_transactions, list)
assert len(test_transactions) > 0

expected_fields = {
    "timestamp", "register_id", "item",
    "price", "currency", "status"
}

assert all(
    expected_fields.issubset(transaction.keys())
    for transaction in test_transactions
)

print("Encoding:", test_encoding)
print("Valid transactions:", len(test_transactions))

display(pd.DataFrame(test_transactions).head())

# Test ISO-8859-1 fallback with a non-ASCII character
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    test_latin1_path = Path(temp_dir) / "latin1_test.log"
    expected_text = "Café transaction"

    test_latin1_path.write_text(
        expected_text,
        encoding="ISO-8859-1"
    )

    latin1_content, latin1_encoding = read_text_file(
        test_latin1_path
    )

    assert latin1_encoding == "ISO-8859-1", \
        "Failed: ISO-8859-1 encoding was not detected."

    assert latin1_content == expected_text, \
        "Failed: File content was not decoded correctly."

    print("\nISO-8859-1 fallback test:")
    print("Encoding:", latin1_encoding)
    print("Content:", latin1_content)


print("PASS: File reading and log parsing")



Encoding: UTF-8
Valid transactions: 80


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
1,2026-10-15 07:02:55,POS_01,Sandwich,7.70,USD,SUCCESS
2,2026-10-15 07:05:45,POS_01,Sandwich,7.43,USD,SUCCESS
3,2026-10-15 07:07:17,POS_01,Cappuccino,3.72,USD,SUCCESS
4,2026-10-15 07:08:34,POS_01,Muffin,3.45,EUR,SUCCESS



ISO-8859-1 fallback test:
Encoding: ISO-8859-1
Content: Café transaction
PASS: File reading and log parsing


In [44]:

# Test recursive file search and active register IDs
test_data_directory = find_data_file("store_config.json").parent

test_log_files = find_log_files(test_data_directory)

with open(
    test_data_directory / "store_config.json",
    "r",
    encoding="utf-8"
) as test_config_file:
    test_config = json.load(test_config_file)

test_active_ids = find_active_register_ids(test_config)

assert isinstance(test_log_files, list)
assert all(isinstance(path, Path) for path in test_log_files)
assert len(test_log_files) > 0
assert isinstance(test_active_ids, set)

print("Log files found:", len(test_log_files))
print("Active register IDs:", sorted(test_active_ids))

display(pd.DataFrame({
    "Log file": [str(path) for path in test_log_files]
}))

print("PASS: Recursive functions")



Log files found: 9
Active register IDs: ['POS_01', 'POS_02', 'POS_03']


,Log file
0,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
1,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
2,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
3,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
4,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
5,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
6,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
7,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
8,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...


PASS: Recursive functions


In [45]:


# Test loading transactions from multiple log files
test_all_transactions = load_all_transactions(test_log_files)

assert isinstance(test_all_transactions, list)
assert len(test_all_transactions) > 0
assert all(
    isinstance(transaction, dict)
    for transaction in test_all_transactions
)

test_transactions_df = pd.DataFrame(test_all_transactions)

print("Total transactions:", len(test_all_transactions))

print("\nTransactions by status:")
display(
    test_transactions_df["status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Count")
)

print("\nSample transactions:")
display(test_transactions_df.head())

print("PASS: Data loading")



Total transactions: 720

Transactions by status:


,Status,Count
0,SUCCESS,467
1,CANCELLED,128
2,REFUND,125



Sample transactions:


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
1,2026-10-15 07:02:55,POS_01,Sandwich,7.70,USD,SUCCESS
2,2026-10-15 07:05:45,POS_01,Sandwich,7.43,USD,SUCCESS
3,2026-10-15 07:07:17,POS_01,Cappuccino,3.72,USD,SUCCESS
4,2026-10-15 07:08:34,POS_01,Muffin,3.45,EUR,SUCCESS


PASS: Data loading


In [46]:

# Test price outlier removal
test_prices = np.array([4.50] * 20 + [999.00])

test_cleaned_prices = remove_price_outliers(test_prices)

assert isinstance(test_cleaned_prices, np.ndarray)
assert test_cleaned_prices.ndim == 1
assert test_cleaned_prices[-1] == 4.50
assert test_prices[-1] == 999.00

print("Price cleaning comparison:")

display(pd.DataFrame({
    "Original price": test_prices,
    "Cleaned price": test_cleaned_prices
}).tail())

# Test refund ratio calculation, including zero sales
test_sales = np.array([100, 50, 20, 0])
test_refunds = np.array([5, 2, 4, 0])

test_ratios = compute_refund_ratio(
    test_sales, test_refunds
)

expected_ratios = (
    test_refunds / (test_sales + 10**-6)
)

assert np.allclose(test_ratios, expected_ratios), \
    "Failed: Refund ratios do not match expected values."

assert np.isfinite(test_ratios).all(), \
    "Failed: Refund ratios contain infinite or NaN values."

assert test_ratios[-1] == 0.0, \
    "Failed: Zero sales and zero refunds should return 0."

print("\nRefund ratio comparison:")

display(pd.DataFrame({
    "Sales": test_sales,
    "Refunds": test_refunds,
    "Refund ratio": test_ratios
}).round(4))

print("PASS: NumPy functions")


Price cleaning comparison:


,Original price,Cleaned price
16,4.5,4.5
17,4.5,4.5
18,4.5,4.5
19,4.5,4.5
20,999.0,4.5



Refund ratio comparison:


,Sales,Refunds,Refund ratio
0,100,5,0.05
1,50,2,0.04
2,20,4,0.20
3,0,0,0.00


PASS: NumPy functions


In [47]:

# Test DataFrame creation
test_df = create_transaction_dataframe(
    test_all_transactions
)

assert isinstance(test_df, pd.DataFrame)
assert isinstance(test_df.index, pd.DatetimeIndex)
assert test_df.index.is_monotonic_increasing

print("Transaction DataFrame:")
display(test_df.head())


# Test NOK price correction
test_currency_df = pd.DataFrame({
    "item": ["Coffee", "Tea", "Latte"],
    "price": [4.0, 5.0, 8.0],
    "currency": ["NOK", "EUR", "NOK"]
})

test_corrected_df = correct_nok_prices(
    test_currency_df
)

nok_mask = test_currency_df["currency"] == "NOK"

assert np.allclose(
    test_corrected_df.loc[nok_mask, "price"],
    test_currency_df.loc[nok_mask, "price"] * 10
)

assert np.allclose(
    test_corrected_df.loc[~nok_mask, "price"],
    test_currency_df.loc[~nok_mask, "price"]
)

print("\nNOK price correction:")

display(pd.DataFrame({
    "Item": test_currency_df["item"],
    "Currency": test_currency_df["currency"],
    "Original price": test_currency_df["price"],
    "Corrected price": test_corrected_df["price"]
}))


# Test currency conversion
test_converted_df = convert_prices_to_nok(
    test_corrected_df
)

assert np.allclose(
    test_converted_df["price"],
    [40.0, 50.0, 80.0]
)

assert (test_converted_df["currency"] == "NOK").all()

print("\nCurrency conversion to NOK:")

display(pd.DataFrame({
    "Item": test_corrected_df["item"],
    "Original currency": test_corrected_df["currency"],
    "Price before conversion": test_corrected_df["price"],
    "Price in NOK": test_converted_df["price"]
}))

print("PASS: Pandas and currency functions")




Transaction DataFrame:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
2026-10-15 07:01:09,POS_02,Cappuccino,3.82,EUR,CANCELLED
2026-10-15 07:01:17,POS_01,Muffin,3.06,EUR,SUCCESS
2026-10-15 07:01:23,POS_02,Muffin,3.84,NOK,SUCCESS
2026-10-15 07:01:56,POS_02,Sandwich,7.17,USD,SUCCESS



NOK price correction:


,Item,Currency,Original price,Corrected price
0,Coffee,NOK,4.0,40.0
1,Tea,EUR,5.0,5.0
2,Latte,NOK,8.0,80.0



Currency conversion to NOK:


,Item,Original currency,Price before conversion,Price in NOK
0,Coffee,NOK,40.0,40.0
1,Tea,EUR,5.0,50.0
2,Latte,NOK,80.0,80.0


PASS: Pandas and currency functions


In [48]:

# Prepare cleaned transaction data
test_report_df = test_df.copy()

test_report_df["price"] = remove_price_outliers(
    test_report_df["price"].to_numpy(copy=True)
)

# Correct NOK price errors
test_report_df = correct_nok_prices(
    test_report_df
)

# Convert currencies before report generation
test_report_df_nok = convert_prices_to_nok(
    test_report_df
)

assert (test_report_df_nok["currency"] == "NOK").all()


# Test hourly revenue
test_hourly_revenue = calculate_hourly_revenue(
    test_report_df_nok
)

assert isinstance(test_hourly_revenue, pd.DataFrame)
assert {"item", "timestamp", "price"}.issubset(
    test_hourly_revenue.columns
)

print("Hourly revenue (NOK):")
display(test_hourly_revenue.head(10))


# Test revenue pivot table
test_revenue_pivot = create_revenue_pivot(
    test_report_df_nok
)

assert isinstance(test_revenue_pivot, pd.DataFrame)
assert list(test_revenue_pivot.columns) == [
    "SUCCESS", "REFUND", "CANCELLED"
]

print("\nRevenue pivot table (NOK):")
display(test_revenue_pivot)


# Test refund ratios by item
test_refund_report = calculate_refund_ratio_by_item(
    test_report_df_nok
)

assert isinstance(test_refund_report, pd.DataFrame)
assert {
    "sales_count", "refund_count", "refund_ratio"
}.issubset(test_refund_report.columns)

assert test_refund_report["refund_ratio"].ge(0).all()

print("\nRefund ratios by item:")
display(test_refund_report.round(4))


# Verify that report totals match
total_transactions = test_report_df_nok["price"].sum()
total_hourly = test_hourly_revenue["price"].sum()
total_pivot = test_revenue_pivot.to_numpy().sum()

assert np.isclose(total_transactions, total_hourly)
assert np.isclose(total_transactions, total_pivot)

print("\nTotal transaction values (NOK):")
print(f"Transactions: {total_transactions:.2f}")
print(f"Hourly report: {total_hourly:.2f}")
print(f"Pivot table:   {total_pivot:.2f}")

print("\nPASS: Report generation")



Hourly revenue (NOK):


,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,2326.7
1,Cappuccino,2026-10-15 08:00:00,2501.6
2,Cappuccino,2026-10-15 09:00:00,375.6
3,Croissant,2026-10-15 07:00:00,2236.6
4,Croissant,2026-10-15 08:00:00,2232.3
5,Croissant,2026-10-15 09:00:00,304.6
6,Espresso,2026-10-15 07:00:00,1586.1
7,Espresso,2026-10-15 08:00:00,1464.4
8,Espresso,2026-10-15 09:00:00,266.8
9,Latte,2026-10-15 07:00:00,2116.7



Revenue pivot table (NOK):


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,3454.6,754.5,994.8
Croissant,3231.4,988.4,553.7
Espresso,1960.3,608.7,748.3
Latte,3275.8,844.2,907.2
Muffin,3181.0,717.9,663.2
Sandwich,5310.1,1686.3,1787.2



Refund ratios by item:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,82,18,0.2195
Croissant,85,25,0.2941
Espresso,65,20,0.3077
Latte,73,19,0.2603
Muffin,90,20,0.2222
Sandwich,72,23,0.3194



Total transaction values (NOK):
Transactions: 31667.60
Hourly report: 31667.60
Pivot table:   31667.60

PASS: Report generation
